# TARANG: Cloud Runs, Cache and Cost Control

Cloud FDTD runs cost FlexCredits. This notebook shows how TARANG makes sure you never pay
twice for the same result and never more than you allowed. Cells 1–4 are local and free;
cell 5 contacts Tidy3D (it needs an API key, `tidy3d configure`) but **charges nothing**:
the job is priced and deleted before it starts. Cell 6, an actual run, is switched off
unless you turn it on.

| Idea | Where it shows up |
|---|---|
| A result is stored under a **fingerprint** of the complete simulation | `sim_fingerprint`, `get_dir_name()` |
| Same simulation again → loaded from disk, no new run | `is_cached()`, `last_run.from_cache` |
| Any change (a dimension, a material, the mesh) → new fingerprint → new run | cell 3 |
| Every run is priced first and stopped above a limit | `max_cost`, `TARANG_MAX_COST`, `CostLimitExceededError` |
| Recompute on purpose | `force_rerun=True` |

In [ ]:
# ---- 1. A small 2D grating coupler (local) ---- #
from pathlib import Path
import numpy as np
from tarang import configure_logging
from tarang.media import get_medium
from tarang.geometries import grating_coupler
from tarang.grating_coupler import GratingConfig

configure_logging("INFO")
SiN = get_medium("si3n4", (1.35, 1.75))
SiO2 = get_medium("sio2", (1.35, 1.75))
Air = get_medium("air")

config = GratingConfig(wavelength_um=1.55, theta_deg=12.0, mfd=10.4, source_x=4.5,
                       source_gap=2.0, core_height=0.8, top_cladding_height=2.0,
                       box_height=4.0, handle_height=8.0, etch_depth=0.5136, period=1.0516,
                       ff=0.5136, n_teeth=22, w_grating=10.4, pol_angle=np.pi / 2, buffer=3.0)
gc = grating_coupler.get_active_simulation(config=config, core_medium=SiN, box_medium=SiO2,
                                           handle_medium=None, top_cladding_medium=Air,
                                           is_3d=False)
out = Path("example_results/cloud_basics")

## 2. Fingerprint and cache folder

The fingerprint is a SHA-256 hash of the complete Tidy3D simulation: geometry, materials,
mesh, sources, monitors. The cache folder is named after the device parameters plus the
start of the fingerprint.

In [ ]:
# ---- 2. Where this result would be stored ---- #
from tarang.utilities.cloud_run import sim_fingerprint

print("fingerprint :", sim_fingerprint(gc.sim)[:16], "...")
print("cache folder:", out / gc.get_dir_name())
print("cached?     :", gc.is_cached(out))

## 3. A change makes a new simulation

A 10 nm different period gives a different fingerprint, so it would be run (and paid
for) separately. The same parameters again give the same fingerprint: that result would be
loaded from disk.

In [ ]:
# ---- 3. Same vs changed geometry ---- #
from dataclasses import replace

same = grating_coupler.get_active_simulation(config=config, core_medium=SiN, box_medium=SiO2,
                                             handle_medium=None, top_cladding_medium=Air,
                                             is_3d=False)
changed = grating_coupler.get_active_simulation(config=replace(config, period=1.0616),
                                                core_medium=SiN, box_medium=SiO2,
                                                handle_medium=None, top_cladding_medium=Air,
                                                is_3d=False)
print("same parameters :", sim_fingerprint(same.sim) == sim_fingerprint(gc.sim))
print("period + 10 nm  :", sim_fingerprint(changed.sim) == sim_fingerprint(gc.sim))

## 4. Setting a cost limit

Two ways, the first one wins:

* `device.max_cost = 0.2` on one device (FlexCredits per run);
* `TARANG_MAX_COST` in the environment, for everything in the session, including sweeps
  that build devices internally.

Batch sweeps such as `parametric_optimization_sweep(..., max_cost=...)` cap the whole
batch; the nested sampler takes `credit_limit` for the whole search.

In [ ]:
# ---- 4. Session-wide limit ---- #
import os
os.environ["TARANG_MAX_COST"] = "1.0"
print("session limit:", os.environ["TARANG_MAX_COST"], "FlexCredits per run")

## 5. The limit in action (needs an API key; charges nothing)

With a deliberately tiny limit, TARANG uploads the simulation, asks Tidy3D for the
estimate, sees it is above the limit, deletes the task **before it starts** and raises
`CostLimitExceededError`.

In [ ]:
# ---- 5. Priced, refused, deleted ---- #
from tarang.utilities.cloud_run import CostLimitExceededError

gc.max_cost = 0.0001
try:
    gc.get_efficiency(output_dir=out)
except CostLimitExceededError as err:
    print("Stopped before starting:", err)
except Exception as err:          # e.g. no API key on this machine
    print(f"{type(err).__name__}: {err}")
finally:
    gc.max_cost = None

## 6. A real run, then a free repeat (optional)

Set `RUN_CLOUD = True` to run the 2D coupler once, capped at 0.1 FlexCredits (the
estimate is printed). The second call is served from the
cache: `last_run.from_cache` is `True` and nothing is uploaded. `force_rerun=True` would
run it again (and charge again).

In [ ]:
# ---- 6. Run once, then reuse ---- #
RUN_CLOUD = False

if RUN_CLOUD:
    gc.max_cost = 0.1
    eff = gc.get_efficiency(output_dir=out)
    print(f"efficiency {eff:.3f}; from cache: {gc.last_run.from_cache}; "
          f"estimate {gc.last_run.estimated_cost:.4f} FlexCredits")
    again = grating_coupler.get_active_simulation(config=config, core_medium=SiN,
                                                  box_medium=SiO2, handle_medium=None,
                                                  top_cladding_medium=Air, is_3d=False)
    again.get_efficiency(output_dir=out)
    print("second request from cache:", again.last_run.from_cache)
else:
    print("Skipped. Set RUN_CLOUD = True to run it (capped at 0.1 FlexCredits).")